What is the causal effect of quitting smoking (qsmk = 1) on weight gain by 1982 (wt82), adjusting for confounders using propensity score matching?

In [113]:
from causallib.datasets import load_nhefs
import pandas as pd
from sklearn.linear_model import LogisticRegression
import numpy as np
from pymatch.Matcher import Matcher
from sklearn.neighbors import NearestNeighbors
import altair as alt
from scipy import stats

pd.set_option('display.max_columns', None)

load data

In [2]:
# Load the dataset
data = load_nhefs()

# Access data, treatment, outcome, and features
X, a, y = data.X, data.a, data.y
dfFull = pd.concat([X,a, y], axis= 1)

In [3]:
dfFull.head()

,age,race,sex,smokeintensity,smokeyrs,wt71,active_1,active_2,education_2,education_3,education_4,education_5,exercise_1,exercise_2,age^2,wt71^2,smokeintensity^2,smokeyrs^2,qsmk,wt82_71
0,42,1,0,30,29,79.04,0,0,0,0,0,0,0,1,1764,6247.3216,900,841,0,-10.093960
1,36,0,0,20,24,58.63,0,0,1,0,0,0,0,0,1296,3437.4769,400,576,0,2.604970
2,56,1,1,20,26,56.81,0,0,1,0,0,0,0,1,3136,3227.3761,400,676,0,9.414486
3,68,1,0,3,53,59.42,1,0,0,0,0,0,0,1,4624,3530.7364,9,2809,0,4.990117
4,40,0,0,20,19,87.09,1,0,1,0,0,0,1,0,1600,7584.6681,400,361,0,4.989251


In [4]:
X.head()

,age,race,sex,smokeintensity,smokeyrs,wt71,active_1,active_2,education_2,education_3,education_4,education_5,exercise_1,exercise_2,age^2,wt71^2,smokeintensity^2,smokeyrs^2
0,42,1,0,30,29,79.04,0,0,0,0,0,0,0,1,1764,6247.3216,900,841
1,36,0,0,20,24,58.63,0,0,1,0,0,0,0,0,1296,3437.4769,400,576
2,56,1,1,20,26,56.81,0,0,1,0,0,0,0,1,3136,3227.3761,400,676
3,68,1,0,3,53,59.42,1,0,0,0,0,0,0,1,4624,3530.7364,9,2809
4,40,0,0,20,19,87.09,1,0,1,0,0,0,1,0,1600,7584.6681,400,361


In [5]:
y.head()

0   -10.093960
1     2.604970
2     9.414486
3     4.990117
4     4.989251
Name: wt82_71, dtype: float64

In [6]:
a.head()

0    0
1    0
2    0
3    0
4    0
Name: qsmk, dtype: int64

In [7]:
dfFull.head()

,age,race,sex,smokeintensity,smokeyrs,wt71,active_1,active_2,education_2,education_3,education_4,education_5,exercise_1,exercise_2,age^2,wt71^2,smokeintensity^2,smokeyrs^2,qsmk,wt82_71
0,42,1,0,30,29,79.04,0,0,0,0,0,0,0,1,1764,6247.3216,900,841,0,-10.093960
1,36,0,0,20,24,58.63,0,0,1,0,0,0,0,0,1296,3437.4769,400,576,0,2.604970
2,56,1,1,20,26,56.81,0,0,1,0,0,0,0,1,3136,3227.3761,400,676,0,9.414486
3,68,1,0,3,53,59.42,1,0,0,0,0,0,0,1,4624,3530.7364,9,2809,0,4.990117
4,40,0,0,20,19,87.09,1,0,1,0,0,0,1,0,1600,7584.6681,400,361,0,4.989251


convert true/false

In [8]:
cols = ['active_1', 'active_2', 'education_2','education_3','education_4', 'education_5', 'exercise_1', 'exercise_2']

def convertTrueFalse(df, cols):

    df_new = df.copy()

    for c in cols:
        df_new[c] = np.where(df_new[c] == True, 1, 0)
    

    return df_new


In [9]:
X = convertTrueFalse(X, cols)

define treatment, outcome, covariates

In [12]:
treat = 'qsmk'
outcome = 'wt82_71'

covs = [
        'age'
        ,'race'
        ,'sex'
        ,'smokeintensity'
        ,'smokeyrs' 
        ,'wt71'
        ,'active_2'
        ,'education_5'
        ,'exercise_2'
        ]

estimate propensity scores

In [13]:
logit = LogisticRegression(max_iter=2000)
logit.fit(dfFull[covs], dfFull[treat])

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [14]:
dfFull['pscore'] = logit.predict_proba(dfFull[covs])[:,1]

perform matching

In [17]:
test = dfFull[dfFull[treat]==1]
control = dfFull[dfFull[treat]==0]

In [40]:
nn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
nn.fit(control[['pscore']])

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",1
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'ball_tree'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [41]:
distances, indices = nn.kneighbors(test[['pscore']])

In [45]:
matched_control = control.iloc[indices.flatten()]
matched_df = pd.concat([test, matched_control])

plots

In [ ]:
def standarized_mean_difference(
                                data
                                ,covars
                                ,treat=treat
                                ):
     
    treated = data[data[treat] == 1]
    control = data[data[treat] == 0]
     
    result = pd.DataFrame()

    smds = []
    treatedMeans = []
    controlMeans = []

    for var in covars:
        mean_treated = treated[var].mean()
        treatedMeans = treatedMeans + [mean_treated]

        mean_control = control[var].mean()
        controlMeans = controlMeans + [mean_control]

        sd_treated = treated[var].std()
        sd_control = control[var].std()

        # smd[var] = abs(mean_treated - mean_control) / np.sqrt((sd_treated**2 + sd_control**2) / 2)
        smd = abs(mean_treated - mean_control) / np.sqrt((sd_treated**2 + sd_control**2) / 2)

        smds = smds + [smd]
    
    result['variable'] = covars
    result['treated mean'] = treatedMeans
    result['control mean'] = controlMeans
    result['standardized mean difference'] = smds
         
    return result


def proportion_difference(
                                data
                                ,covars
                                ,treat=treat
                                ):
     
    treated = data[data[treat] == 1]
    control = data[data[treat] == 0]
     
    result = pd.DataFrame()

    diffs = []
    treatedMeans = []
    controlMeans = []

    for var in covars:
        mean_treated = treated[var].mean()
        treatedMeans = treatedMeans + [mean_treated]

        mean_control = control[var].mean()
        controlMeans = controlMeans + [mean_control]

        sd_treated = treated[var].std()
        sd_control = control[var].std()

        diff = abs(mean_treated - mean_control)

        diffs = diffs + [diff]
    
    result['variable'] = covars
    result['treated mean'] = treatedMeans
    result['control mean'] = controlMeans
    result['mean difference'] = diffs
         
    return result

In [96]:
smd_before_matching = standarized_mean_difference(dfFull, ['age', 'smokeintensity', 'smokeyrs', 'wt71'])
smd_before_matching['before_after'] = 'before'

smd_after_matching = standarized_mean_difference(matched_df, ['age', 'smokeintensity', 'smokeyrs', 'wt71'])
smd_after_matching['before_after'] = 'after'

beforeAfter = pd.concat([smd_before_matching,smd_after_matching])

In [99]:
beforeAfter

,variable,treated mean,control mean,standardized mean difference,before_after
0,age,46.173697,42.788478,0.281981,before
1,smokeintensity,18.602978,21.191745,0.216675,before
2,smokeyrs,26.032258,24.087704,0.158918,before
3,wt71,72.354888,70.302837,0.133216,before
0,age,46.173697,45.143921,0.084403,after
1,smokeintensity,18.602978,18.181141,0.037500,after
2,smokeyrs,26.032258,24.483871,0.124576,after
3,wt71,72.354888,72.602060,0.015958,after


In [100]:
chart = alt.Chart(beforeAfter).mark_circle(size=100).encode(
                x='standardized mean difference',
                y=alt.Y('variable:N', axis=alt.Axis(grid=True)),
                color=alt.Color('before_after:N', legend=alt.Legend(title=None))
            ).properties(
                title='Before and After',
                height=200
            )\
            .configure_axis(grid=False)

chart

alt.Chart(...)

In [106]:
smd_before_matching = proportion_difference(dfFull, ['race', 'sex', 'active_2', 'education_5', 'exercise_2'])
smd_before_matching['before_after'] = 'before'

smd_after_matching = proportion_difference(matched_df, ['race', 'sex', 'active_2', 'education_5', 'exercise_2'])
smd_after_matching['before_after'] = 'after'

beforeAfter = pd.concat([smd_before_matching,smd_after_matching])

In [107]:
beforeAfter

,variable,treated mean,control mean,mean difference,before_after
0,race,0.089330,0.146174,0.056844,before
1,sex,0.454094,0.533964,0.079870,before
2,active_2,0.111663,0.089424,0.022239,before
3,education_5,0.153846,0.098882,0.054964,before
4,exercise_2,0.406948,0.379192,0.027756,before
0,race,0.089330,0.104218,0.014888,after
1,sex,0.454094,0.481390,0.027295,after
2,active_2,0.111663,0.084367,0.027295,after
3,education_5,0.153846,0.198511,0.044665,after
4,exercise_2,0.406948,0.397022,0.009926,after


In [108]:
chart = alt.Chart(beforeAfter).mark_circle(size=100).encode(
                x='mean difference',
                y=alt.Y('variable:N', axis=alt.Axis(grid=True)),
                color=alt.Color('before_after:N', legend=alt.Legend(title=None))
            ).properties(
                title='Before and After',
                height=200
            )\
            .configure_axis(grid=False)

chart

alt.Chart(...)

estimate att

In [112]:
treated = matched_df[matched_df[treat] == 1]
control = matched_df[matched_df[treat] == 0]

att = treated[outcome].mean() - control[outcome].mean()
att


3.108292318957817

In [116]:
stats.ttest_ind(treated[outcome], control[outcome], equal_var=False)

TtestResult(statistic=5.719201976564839, pvalue=1.5508527289774764e-08, df=743.2613043809507)